# AIDER Pipeline Execution
This notebook unifies the image standardization, synthetic metadata generation, and ground truth augmentation steps into a single sequential workflow.

In [1]:
import os
import sys
import glob
import hashlib
import random
import time
import json
import math
import re
from datetime import datetime, timezone
from collections import defaultdict

try:
    from google import genai
    from google.genai import types
    from PIL import Image
    from dotenv import load_dotenv
    from tqdm import tqdm
except ImportError:
    import subprocess
    print("Missing dependencies. Installing google-genai, pillow, python-dotenv, tqdm...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "google-genai", "pillow", "python-dotenv", "tqdm"])
    from google import genai
    from google.genai import types
    from PIL import Image
    from dotenv import load_dotenv
    from tqdm import tqdm


# --- PATH CONFIGURATION ---
# In a notebook inside support/, __file__ is not defined. We use os.getcwd() 
# assuming Jupyter is started in project root or support directory.
# --- PATH CONFIGURATION ---
SUPPORT_DIR = os.getcwd()
if os.path.basename(SUPPORT_DIR) != 'support':
    SUPPORT_DIR = os.path.join(SUPPORT_DIR, 'support')
PROJECT_ROOT = os.path.dirname(SUPPORT_DIR)

SOURCE_IMG_DIR = os.path.join(PROJECT_ROOT, "data", "AIDER")
PROCESSED_IMG_DIR = os.path.join(PROJECT_ROOT, "data", "aider_processed_images")
OUTPUT_LABEL_DIR = os.path.join(PROJECT_ROOT, "data", "aider_processed_labels")
TRACKER_FILE = os.path.join(PROJECT_ROOT, "data", "aider_processed_tracker.txt")

# --- TEAM DELEGATION CONFIGURATION ---
TOTAL_WORKERS = 5  # Total number of people running the script (Abrar, Aryan, Ridita, Tamanna, Noman)
WORKER_ID = 0      # 0=You, 1=Aryan, 2=Ridita, 3=Tamanna, 4=Noman (Change this before running!)

DATASET_OUT_PATH = os.path.join(PROJECT_ROOT, "dataset", f"aider_train_dataset_worker_{WORKER_ID}.jsonl")

# Create output directories if they don't exist
os.makedirs(PROCESSED_IMG_DIR, exist_ok=True)
os.makedirs(OUTPUT_LABEL_DIR, exist_ok=True)
os.makedirs(os.path.dirname(DATASET_OUT_PATH), exist_ok=True)

# --- SHARED CONFIGURATION ---
EXCLUDE_CLASSES = ['normal']  # Classes to skip during metadata & ground truth generation
MODELS = [
    'gemini-3.5-flash',
    'gemini-2.5-flash'
]

# --- API KEY CONFIGURATION ---
load_dotenv(os.path.join(PROJECT_ROOT, '.env'))
GEMINI_API_KEYS = [v.strip() for k, v in os.environ.items() 
                   if k.startswith("GEMINI_API_KEY") and v.strip()]

if not GEMINI_API_KEYS:
    print("ERROR: No GEMINI_API_KEY found in the .env configuration.")
else:
    # Set default for SDKs that check os.environ directly
    os.environ['GEMINI_API_KEY'] = GEMINI_API_KEYS[0]


## 1. Image Standardization
Filters, resizes, and converts raw AIDER images to standard JPEGs.

In [2]:
# Image standardization parameters (matching xBD pipeline conventions)
MAX_EDGE = 1280  # Cap longest edge for VRAM safety on T4 GPUs
MIN_EDGE = 256   # Minimum resolution gate
JPEG_QUALITY = 95

def load_processed_history():
    """Loads the set of already processed image base names from the tracker."""
    if not os.path.exists(TRACKER_FILE):
        return set()
    processed = set()
    with open(TRACKER_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                parts = line.split('|')
                base = parts[0].strip()
                processed.add(base)
    return processed

def append_to_history(original, output, md5_hash, status):
    """Appends a processed image record to the tracker file."""
    timestamp = datetime.now(timezone.utc).isoformat()
    line = f"{original} | {output} | {md5_hash} | {timestamp} | {status}"
    with open(TRACKER_FILE, 'a', encoding='utf-8') as f:
        f.write(line + '\n')

def standardize_image(input_path, output_path, max_edge=MAX_EDGE):
    """
    Standardizes an image for VLM fine-tuning:
    - Skips naturally grayscale imagery.
    - Converts RGBA/P/etc. to RGB.
    - Caps longest edge to max_edge (preserving aspect ratio).
    - Rejects images below MIN_EDGE after scaling.
    - Saves as optimized JPEG.
    - Verifies output integrity.
    - Computes MD5 hash for deduplication audits.
    """
    try:
        with Image.open(input_path) as img:
            # Skip naturally grayscale imagery
            if img.mode == 'L':
                return False, "skipped_grayscale", "NONE"

            # Drop alpha/multispectral channels
            if img.mode != 'RGB':
                img = img.convert('RGB')

            width, height = img.size
            # Dynamically downscale if it exceeds max_edge, preserving aspect ratio
            if max(width, height) > max_edge:
                scaling_factor = max_edge / float(max(width, height))
                new_size = (int(width * scaling_factor), int(height * scaling_factor))
                img = img.resize(new_size, Image.Resampling.LANCZOS)

            # Minimum resolution gate
            if min(img.size) < MIN_EDGE:
                return False, "skipped_too_small", "NONE"

            # Save cleanly as JPEG
            img.save(output_path, "JPEG", quality=JPEG_QUALITY, optimize=True)

            # Integrity verification
            Image.open(output_path).verify()

            # Compute MD5 hash
            with open(output_path, "rb") as f:
                md5_hash = hashlib.md5(f.read()).hexdigest()

            return True, "success", md5_hash

    except Exception as e:
        print(f"  ❌ Error processing {input_path}: {e}")
        return False, "failed_error", "NONE"

def run_standardization():
    print(f"🔍 Scanning {SOURCE_IMG_DIR} for AIDER images...")

    search_pattern = os.path.join(SOURCE_IMG_DIR, "**", "*.*")
    all_files = glob.glob(search_pattern, recursive=True)
    img_files = [f for f in all_files if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    if not img_files:
        print("No images found! Check that data/AIDER/ contains class subdirectories.")
        return

    random.seed(42)
    random.shuffle(img_files)

    print(f"Found {len(img_files)} images across AIDER classes.")
    processed_history = load_processed_history()
    print(f"Already processed in previous runs: {len(processed_history)}")

    successful = 0
    skipped_existing = 0
    failed = 0

    for img_path in img_files:
        parent_dir = os.path.basename(os.path.dirname(img_path))
        filename = os.path.basename(img_path)

        if parent_dir not in filename:
            out_name = f"{parent_dir}_{os.path.splitext(filename)[0]}.jpg"
        else:
            out_name = f"{os.path.splitext(filename)[0]}.jpg"

        if out_name in processed_history:
            skipped_existing += 1
            continue

        output_path = os.path.join(PROCESSED_IMG_DIR, out_name)
        success, status, md5_hash = standardize_image(img_path, output_path)

        append_to_history(out_name, out_name if success else "NONE", md5_hash, status)
        processed_history.add(out_name)

        if success:
            successful += 1
            if successful % 100 == 0:
                print(f"  [{successful}] Processed: {out_name}")
        else:
            failed += 1

    print(f"\n{'='*60}")
    print(f"📊 AIDER STANDARDIZATION REPORT")
    print(f"{'='*60}")
    print(f"  Total scanned         : {len(img_files)}")
    print(f"  Already processed     : {skipped_existing}")
    print(f"  Newly processed       : {successful}")
    print(f"  Failed / Skipped      : {failed}")
    print(f"{'='*60}")

# Execute standardization
run_standardization()


🔍 Scanning d:\CSE499AB_project\data\AIDER for AIDER images...
Found 6433 images across AIDER classes.
Already processed in previous runs: 6433

📊 AIDER STANDARDIZATION REPORT
  Total scanned         : 6433
  Already processed     : 6433
  Newly processed       : 0
  Failed / Skipped      : 0


## 2. Synthetic Metadata Generation
Synthesizes xBD-equivalent JSON metadata for processed AIDER images.

In [ ]:
TARGET_SYNTHESIS_COUNT = 100  # Set to 50, 100, or len(img_files)

# VERY IMPORTANT: Lock random seed so the execution queue is identical every run
# This prevents the queue from shifting if the cell is interrupted and restarted.
random.seed(42)


SYSTEM_PROMPT = """You are a geospatial analysis AI. 
Your task is to analyze this aerial/disaster image and synthesize a structured JSON metadata report. 
We need to emulate an instance-segmentation format (xBD schema).
Even though this is an image-level classification dataset, look at the image and identify 1 to 5 distinct "features" (e.g., a building, a road, a tree line, a vehicle). 
For each feature you identify, assign a severity subtype from this exact list:
['destroyed', 'major-damage', 'minor-damage', 'no-damage', 'un-classified']

Respond ONLY with a valid JSON object using the exact schema below. Do not include markdown blocks (```json).

{
  "metadata": {
    "dataset": "AIDER",
    "disaster_type": "<fire|flood|traffic_incident|collapsed_building|normal>"
  },
  "features": {
    "xy": [
      {
        "properties": {
          "feature_type": "<e.g., building, road, vehicle>",
          "subtype": "<severity_from_list_above>"
        }
      }
    ]
  }
}"""

def run_metadata_synthesis():
    img_files = glob.glob(os.path.join(PROCESSED_IMG_DIR, "*.jpg"))
    if not img_files:
        print(f"No images found in {PROCESSED_IMG_DIR}. Please run standardization first!")
        return

    print("Building inventory for metadata synthesis...")
    inventory = defaultdict(list)
    for img_path in img_files:
        base_name = os.path.basename(img_path)
        cls_name = base_name.split('_image')[0] if '_image' in base_name else 'unknown'
        if cls_name not in EXCLUDE_CLASSES:
            inventory[cls_name].append(img_path)
            
    active_classes = list(inventory.keys())
    if not active_classes:
        print("No active classes found after exclusion filters.")
        return

    per_class_quota = math.ceil(TARGET_SYNTHESIS_COUNT / len(active_classes))
    print(f"Targeting {TARGET_SYNTHESIS_COUNT} total labels (~{per_class_quota} per active class).")

    execution_queue = []
    for cls in active_classes:
        paths = inventory[cls]
        random.shuffle(paths)
        
        # SHARDING: Slice only this worker's share of this specific class
        my_shard = paths[WORKER_ID::TOTAL_WORKERS]
        
        selected = my_shard[:per_class_quota]
        execution_queue.extend(selected)


    random.shuffle(execution_queue)
    execution_queue = execution_queue[:TARGET_SYNTHESIS_COUNT]

    print(f"Synthesizing pseudo-xBD JSON metadata for {len(execution_queue)} AIDER images...")
    
    current_key_idx = 0
    client = genai.Client(api_key=GEMINI_API_KEYS[current_key_idx])
    current_model_idx = 0
    current_model = MODELS[current_model_idx]

    processed_count = 0
    for img_path in tqdm(execution_queue, desc="Synthesizing Labels"):
        base_name = os.path.basename(img_path).replace('.jpg', '')
        json_out_path = os.path.join(OUTPUT_LABEL_DIR, f"{base_name}.json")
        
        if os.path.exists(json_out_path):
            processed_count += 1
            continue
            
        success = False
        max_retries = (len(GEMINI_API_KEYS) * len(MODELS)) + 3
        retries = 0

        context_prompt = f"System Instruction:\n{SYSTEM_PROMPT}\n\nGround Truth Hint: The image filename is '{base_name}'."

        while retries < max_retries and not success:
            try:
                img = Image.open(img_path)
                response = client.models.generate_content(
                    model=current_model,
                    contents=[img, context_prompt],
                    config=types.GenerateContentConfig(
                        temperature=0.2,
                    )
                )
                
                output = response.text.strip()
                if output.startswith("```json"):
                    output = output.replace("```json", "").replace("```", "").strip()
                elif output.startswith("```"):
                    output = output.replace("```", "").strip()
                    
                json_data = json.loads(output)
                
                with open(json_out_path, 'w', encoding='utf-8') as f:
                    json.dump(json_data, f, indent=2)
                    
                success = True
                processed_count += 1
                time.sleep(2) # rate limit cooldown

            except Exception as e:
                err = str(e).lower()
                if "503" in err or "unavailable" in err:
                    time.sleep(15)
                    retries += 1
                elif "429" in err or "quota" in err or "api_key_invalid" in err:
                    if "requestsperday" in err or "free_tier_requests" in err or "api_key_invalid" in err:
                        current_model_idx += 1
                        if current_model_idx >= len(MODELS):
                            current_key_idx = (current_key_idx + 1) % len(GEMINI_API_KEYS)
                            client = genai.Client(api_key=GEMINI_API_KEYS[current_key_idx])
                            current_model_idx = 0
                        current_model = MODELS[current_model_idx]
                    elif "requestsperminute" in err:
                        time.sleep(60)
                    elif "tokensperminute" in err:
                        time.sleep(30)
                    else:
                        current_key_idx = (current_key_idx + 1) % len(GEMINI_API_KEYS)
                        client = genai.Client(api_key=GEMINI_API_KEYS[current_key_idx])
                        time.sleep(2)
                    retries += 1
                elif "json.decoder.jsondecodeerror" in str(type(e)).lower():
                    retries += 1
                    time.sleep(2)
                else:
                    print(f"\nUnhandled Error parsing {img_path}: {e}")
                    break

        if not success:
            print(f"\nFailed to process {base_name} after {retries} retries. Cooling down...")
            time.sleep(30)
            
    print(f"\nMetadata Synthesis Complete. Synthesized {processed_count} JSON labels in {OUTPUT_LABEL_DIR}")

if GEMINI_API_KEYS:
    run_metadata_synthesis()


Building inventory for metadata synthesis...
Targeting 100 total labels (~25 per active class).
Synthesizing pseudo-xBD JSON metadata for 100 AIDER images...


Synthesizing Labels:   1%|          | 1/100 [00:21<34:39, 21.01s/it]


KeyboardInterrupt: 

## 3. Ground Truth Generation
Augments synthesized metadata + processed imagery into tactical rescue QLoRA pairs.

In [ ]:
AUGMENTATION_MULTIPLIER = 1  # 1 = 1:1 generation. Set to >1 to augment a small label set.

# VERY IMPORTANT: Lock random seed so the execution queue is identical every run
random.seed(42)


SYSTEM_INSTRUCTION = """
You are an expert disaster response and structural engineering analyst. 
You will be provided with an aerial post-disaster image AND localized metadata annotations. 
Your task is to combine the visual evidence from the image with the hard numbers from the metadata to generate a highly professional, concise, and tactical rescue report. 
Do not hallucinate hazards, but explicitly mention severe visual hazards (like fires, floods, collapsed structures, or vehicle wreckage) even if the metadata does not explicitly list them.

You must strictly output your assessment following this schema without deviation:
### 1. Priority Zones (Geospatial Mapping)
[Identify areas based on the specific prompt perspective.]
### 2. Structural Damage & Collapse Characterization
[Classify the observed architectural failures based on the provided data.]
### 3. Hazard Avoidance & Logistics Constraints
[Highlight secondary tactical risks visible in the image or noted in the data.]

Constraint: Do not include introductory or concluding pleasantries. Maintain an authoritative, objective, and operational tone. Keep your response highly concise (maximum 300 words).
"""

AUGMENTATION_PROMPTS = {
    "structural": "Focus exclusively on building collapse modes and survivor void spaces. Identify pancake collapses, lean-over failures, V-space formations, and estimate the likelihood of survivable voids beneath debris.",
    "logistics": "Focus exclusively on rescue force ingress and egress. Identify blocked roads, bridge integrity, landing zones for helicopters or boats, and the safest approach corridors for ground teams.",
    "environmental": "Focus exclusively on secondary hazard propagation. Identify active or potential fires, flood extent, chemical spills, vehicle fuel leakage, or structural instability that poses risk to rescue personnel.",
    "triage": "Focus exclusively on survivor prioritization. Based on building density, damage severity, and visible signs of recent occupancy, rank zones by expected survivor concentration and medical urgency.",
}
PERSPECTIVE_KEYS = ["structural", "triage", "logistics", "environmental"]

DISASTER_PROMPTS = {
    'collapsed_building': "Focus on collapse mode (pancake, lean-over, V-space), void identification, and column/stairwell proximity.",
    'fire':              "Focus on burn perimeter, structure integrity after thermal stress, and ember-cast secondary ignition zones.",
    'flood':             "Focus on water ingress depth estimation, roof refuge identification, and waterborne access routes.",
    'flooded_areas':     "Focus on water ingress depth estimation, roof refuge identification, and waterborne access routes.",
    'traffic_incident':  "Focus on vehicle collision severity, road blockage extent, fuel spill hazards, and evacuation route viability.",
    'normal':            "Focus on baseline structural integrity assessment and confirm absence of visible damage indicators.",
}

def get_disaster_emphasis(dtype: str) -> str:
    dtype = dtype.lower()
    for k, v in DISASTER_PROMPTS.items():
        if k in dtype:
            return f" {v}"
    return ""

SEVERITY_ORDER = ['destroyed', 'major-damage', 'minor-damage', 'no-damage', 'un-classified']

def get_max_severity(features: list) -> str:
    present = {f.get('properties', {}).get('subtype', 'un-classified') for f in features}
    for level in SEVERITY_ORDER:
        if level in present:
            return level
    return 'un-classified'

def get_multiplier(available: int, needed: int) -> int:
    ratio = needed / max(available, 1)
    if ratio <= 1: return 1
    elif ratio <= 2: return 2
    elif ratio <= 3: return 3
    else: return 4

def parse_synthetic_json(json_path):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    metadata = data.get('metadata', {})
    dtype = metadata.get('disaster_type', 'unknown')
    
    # Robust extraction to handle LLM schema hallucinations
    features_obj = data.get('features', {})
    if isinstance(features_obj, dict):
        features = features_obj.get('xy', [])
    elif isinstance(features_obj, list):
        features = features_obj
    else:
        features = []
        
    total_features = len(features)

    damage_counts = {'no-damage': 0, 'minor-damage': 0, 'major-damage': 0, 'destroyed': 0, 'un-classified': 0}
    for feature in features:
        damage_level = feature.get('properties', {}).get('subtype', 'un-classified')
        if damage_level in damage_counts:
            damage_counts[damage_level] += 1
    summary = (
        f"Disaster Type: {dtype}\n"
        f"Total Features Detected: {total_features}\n"
        f"Damage Assessment: {damage_counts['destroyed']} destroyed, "
        f"{damage_counts['major-damage']} major damage, "
        f"{damage_counts['minor-damage']} minor damage, "
        f"{damage_counts['no-damage']} intact."
    )
    max_sev = get_max_severity(features)
    return summary, dtype, max_sev

def passes_qa_gates(response_text: str) -> bool:
    has_1 = any(h in response_text for h in ["### 1.", "## 1.", "# 1.", "1. Priority"])
    has_2 = any(h in response_text for h in ["### 2.", "## 2.", "# 2.", "2. Structural"])
    has_3 = any(h in response_text for h in ["### 3.", "## 3.", "# 3.", "3. Hazard"])
    if not (has_1 and has_2 and has_3):
        return False
    if len(response_text.split()) < 120:
        return False
    first_person_pattern = re.compile(
        r"\b(i)(?!\.e\b)\b|\b(i'm|i cannot|as an ai|i apologize|i'm sorry)\b", re.IGNORECASE
    )
    if first_person_pattern.search(response_text):
        return False
    return True

def load_existing_progress():
    processed_images = set()
    if os.path.exists(DATASET_OUT_PATH):
        with open(DATASET_OUT_PATH, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    try:
                        row = json.loads(line)
                        perspective = row.get("meta", {}).get("perspective", "default")
                        dedup_key = f"{row['image']}::{perspective}"
                        processed_images.add(dedup_key)
                    except Exception:
                        continue
    return processed_images

def run_ground_truth_generation():
    json_files = glob.glob(os.path.join(OUTPUT_LABEL_DIR, "*.json"))
    if not json_files:
        print("No JSON files found! Run metadata synthesis first.")
        return

    raw_inventory = defaultdict(list)
    for jp in json_files:
        _, dtype, _ = parse_synthetic_json(jp)
        raw_inventory[dtype].append(jp)

    inventory = {k: v for k, v in raw_inventory.items() if k not in EXCLUDE_CLASSES}
    if not inventory:
        print("ERROR: All classes were excluded!")
        return

    disaster_classes = list(inventory.keys())
    valid_label_count = sum(len(paths) for paths in inventory.values())
    global_target = valid_label_count * AUGMENTATION_MULTIPLIER
    per_class_quota = math.ceil(global_target / max(len(disaster_classes), 1))
    
    processed_history = load_existing_progress()
    current_count = len(processed_history)

    print(f"\nCurrent AIDER progress: {current_count} augmented samples already in dataset.")
    if current_count >= global_target:
        print(f"Dynamic target of {global_target} samples already met! Exiting.")
        return

    execution_queue = []
    for dtype in disaster_classes:
        available_paths = inventory[dtype]
        random.shuffle(available_paths)

        available_count = len(available_paths)
        if available_count == 0:
            continue

        multiplier = get_multiplier(available_count, per_class_quota)
        images_to_take = min(available_count, per_class_quota)

        selected_paths = available_paths[:images_to_take]


        for p in selected_paths:
            execution_queue.append({
                'json_path': p,
                'disaster': dtype,
                'multiplier': multiplier
            })

    random.shuffle(execution_queue)
    
    api_keys = [k.strip() for k in GEMINI_API_KEYS if k and k.strip()]
    current_key_idx = 0
    client = genai.Client(api_key=api_keys[current_key_idx])
    current_model_idx = 0
    current_model = MODELS[current_model_idx]

    with open(DATASET_OUT_PATH, 'a', encoding='utf-8') as f_out:
        for item in execution_queue:
            if current_count >= global_target:
                break

            json_path = item['json_path']
            base_name = os.path.basename(json_path).replace('.json', '')
            image_path = os.path.join(PROCESSED_IMG_DIR, f"{base_name}.jpg")
            image_reference = f"aider_processed_images/{base_name}.jpg"

            if not os.path.exists(image_path): continue
            metadata_summary, dtype, msev = parse_synthetic_json(json_path)

            for m in range(item['multiplier']):
                if current_count >= global_target: break
                perspective_key = PERSPECTIVE_KEYS[m]
                dedup_key = f"{image_reference}::{perspective_key}"
                if dedup_key in processed_history: continue

                base_instruction = AUGMENTATION_PROMPTS[perspective_key]
                disaster_emphasis = get_disaster_emphasis(dtype)
                prompt_text = f"{base_instruction}{disaster_emphasis}\n\nMetadata Annotations:\n{metadata_summary}"
                instruction = base_instruction

                max_retries = (len(api_keys) * len(MODELS)) + 3
                retries = 0
                success = False

                while retries < max_retries and not success:
                    try:
                        img = Image.open(image_path)
                        response = client.models.generate_content(
                            model=current_model,
                            contents=[img, prompt_text],
                            config=types.GenerateContentConfig(
                                system_instruction=SYSTEM_INSTRUCTION,
                                temperature=0.3,
                                safety_settings=[
                                    types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT, threshold=types.HarmBlockThreshold.BLOCK_NONE),
                                    types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_HARASSMENT, threshold=types.HarmBlockThreshold.BLOCK_NONE),
                                    types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH, threshold=types.HarmBlockThreshold.BLOCK_NONE),
                                    types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT, threshold=types.HarmBlockThreshold.BLOCK_NONE),
                                ]
                            )
                        )
                        ground_truth = response.text.strip()
                        
                        if not passes_qa_gates(ground_truth):
                            retries += 1
                            time.sleep(2)
                            continue

                        jsonl_row = {
                            "image": image_reference,
                            "instruction": instruction,
                            "response": ground_truth,
                            "meta": {
                                "dataset": "AIDER", "disaster_type": dtype, "max_severity": msev,
                                "perspective": perspective_key, "multiplier_applied": item['multiplier'] > 1,
                                "model": current_model
                            }
                        }
                        f_out.write(json.dumps(jsonl_row) + '\n')
                        f_out.flush()
                        processed_history.add(dedup_key)
                        current_count += 1
                        print(f"[{current_count}/{global_target}] SUCCESS: {base_name} [{perspective_key}]")
                        success = True
                        time.sleep(4.3)
                    except Exception as e:
                        err = str(e).lower()
                        if "503" in err or "unavailable" in err:
                            time.sleep(15)
                            retries += 1
                        elif "429" in err or "quota" in err:
                            if "requestsperday" in err or "free_tier" in err:
                                current_model_idx += 1
                                if current_model_idx >= len(MODELS):
                                    current_key_idx = (current_key_idx + 1) % len(api_keys)
                                    client = genai.Client(api_key=api_keys[current_key_idx])
                                    current_model_idx = 0
                                current_model = MODELS[current_model_idx]
                            elif "requestsperminute" in err: time.sleep(60)
                            elif "tokensperminute" in err: time.sleep(30)
                            else:
                                current_key_idx = (current_key_idx + 1) % len(api_keys)
                                client = genai.Client(api_key=api_keys[current_key_idx])
                                time.sleep(2)
                            retries += 1
                        else:
                            print(f"ERROR: {e}")
                            break
                if not success:
                    time.sleep(45)

    print(f"\nExecution terminated. Current Total: {current_count} rows written to {DATASET_OUT_PATH}")

if GEMINI_API_KEYS:
    run_ground_truth_generation()



Current AIDER progress: 69 augmented samples already in dataset.
[70/117] SUCCESS: flooded_areas_flood_image0526 [structural]
[71/117] SUCCESS: fire_image0271 [triage]
[72/117] SUCCESS: collapsed_building_image0407 [structural]
[73/117] SUCCESS: collapsed_building_image0407 [triage]
[74/117] SUCCESS: fire_image0171 [structural]
[75/117] SUCCESS: fire_image0171 [triage]
[76/117] SUCCESS: traffic_incident_image0022 [structural]
[77/117] SUCCESS: flooded_areas_flood_image0510 [structural]
[78/117] SUCCESS: collapsed_building_image0467 [structural]
[79/117] SUCCESS: collapsed_building_image0467 [triage]
[80/117] SUCCESS: fire_image0235 [structural]
[81/117] SUCCESS: fire_image0235 [triage]
[82/117] SUCCESS: traffic_incident_image0301 [structural]
[83/117] SUCCESS: flooded_areas_flood_image0028 [structural]
[84/117] SUCCESS: traffic_incident_image0236 [structural]
[85/117] SUCCESS: traffic_incident_image0063 [structural]
[86/117] SUCCESS: traffic_incident_image0093 [structural]
[87/117] SU